In [1]:
import torch
import numpy as np
from pathlib import Path
ATTACK_WINDOWS = Path("..") / ".." / "Data" / "Attack_Windows"
DATA_DIR = Path("embeddings")

In [2]:
GAP_WINDOWS = 15

def split_attack_windows(file_name): 
    
    # Load the attack window files
    data = torch.load(ATTACK_WINDOWS / f"{file_name}.pt", map_location="cpu", weights_only=False)
    
    # Extract the features from the file 
    windows = data["features"]
    labels = data["labels"]
    
    # Split the windows into train and test sets based on the 70% and 30% split 
    split_index = int(len(windows) * 0.70)
    
    # Split the train window from the beginnign of the file to the split index = 70% 
    train_windows = windows[:split_index]
    train_labels = labels[:split_index]
    
    # Split the rest of the window from the split index + 15 window gap = last part of the window 
    # which corresponds to the 30% of the file 
    test_windows = windows[split_index + GAP_WINDOWS:]
    test_labels = labels[split_index + GAP_WINDOWS:]
    
    return train_windows, train_labels, test_windows, test_labels

In [3]:
attack_files = [
    ATTACK_WINDOWS / "DoS_attack.pt",
    ATTACK_WINDOWS / "Steering_angle_attack.pt",
    ATTACK_WINDOWS / "Fuzzing_random_IDs.pt",
    ATTACK_WINDOWS / "EMS_replay_attack.pt",
]

def baseline(windows):
    windows = windows[:, :, 1:]
    mean_feat = windows.mean(dim=1)
    max_feat = windows.max(dim=1).values
    std_feat = windows.std(dim=1)
    return torch.cat([mean_feat, max_feat, std_feat], dim=1)

X_train_list, X_test_list, y_train_list, y_test_list = [], [], [], []

for path in attack_files:
    file_name = path.stem
    train_windows, train_labels, test_windows, test_labels = split_attack_windows(file_name)

    X_train_list.append(baseline(train_windows))
    y_train_list.append(train_labels)
    X_test_list.append(baseline(test_windows))
    y_test_list.append(test_labels)

    print(file_name, "| Train:", len(train_windows), "| Test:", len(test_windows))

DoS_attack | Train: 61665 | Test: 26414
Steering_angle_attack | Train: 44177 | Test: 18918
Fuzzing_random_IDs | Train: 84513 | Test: 36205
EMS_replay_attack | Train: 45075 | Test: 19303


In [4]:
# Create the numpy arrays 
X_train = torch.cat(X_train_list).numpy()
y_train = torch.cat(y_train_list).numpy()
X_test = torch.cat(X_test_list).numpy()
y_test = torch.cat(y_test_list).numpy()

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("NaN:", np.isnan(X_train).any(), np.isnan(X_test).any())

X_train: (235430, 30) | X_test: (100840, 30)
NaN: False False


In [5]:
# Save the embeddings
np.save(DATA_DIR / "X_train_baseline.npy", X_train)
np.save(DATA_DIR / "y_train_baseline.npy", y_train)
np.save(DATA_DIR / "X_test_baseline.npy", X_test)
np.save(DATA_DIR / "y_test_baseline.npy", y_test)
print("Saved to:", DATA_DIR)

Saved to: embeddings
